# Task decomposition end to end

This notebook is a reusable, stage-by-stage evaluation surface for the canonical task-decomposition APIs. Provider calls are made only by an explicit button after the opt-in checkbox is selected.

After installing this update, restart the kernel and clear all outputs once, then run the notebook from the top. Result panels replace their HTML contents directly; repeated clicks run new experiments. Rerunning the controls cell retires its previous widgets and callbacks.


## 1. Setup and provider configuration

Imports and provider construction are centralized here. Credentials come from the existing environment and are never displayed.

In [1]:
from copy import deepcopy
from datetime import datetime, timezone
from decimal import Decimal
import hashlib, json, os, time
from pathlib import Path
from html import escape
from typing import Any

try:
    import ipywidgets as widgets
    import pandas as pd
    from IPython.display import Markdown, clear_output, display
except ImportError as exc:
    raise ImportError("This notebook requires pandas, ipywidgets, and IPython. Install the project notebook dependencies.") from exc

from task_decomposition import (
    SupportWorkMode, TaskDecompositionRequest, TaskReference,
    TransformationDecompositionRequest, decompose,
    decompose_task, decompose_transformation, estimate_transformation,
)

PROVIDER_ENV = {"OpenAI": "OPENAI_API_KEY", "Gemini": "GEMINI_API_KEY", "DeepSeek": "DEEPSEEK_API_KEY"}
MODEL_OPTIONS = {
    "OpenAI": [("Provider default", ""), ("gpt-6-luna", "gpt-6-luna"), ("gpt-6.1-sol", "gpt-6.1-sol"), ("gpt-6-astra", "gpt-6-astra"), ("gpt-5.5", "gpt-5.5"), ("gpt-4o-mini", "gpt-4o-mini")],
    "Gemini": [("Provider default", ""), ("gemini-3.5-flash-lite", "gemini-3.5-flash-lite"), ("gemini-3.8-flash", "gemini-3.8-flash"), ("gemini-4-argon", "gemini-4-argon")],
    "DeepSeek": [("Provider default (V4.1 Flash)", ""), ("DeepSeek V4.1 Flash (current)", "deepseek-flash"), ("DeepSeek V4 Pro (legacy comparison)", "deepseek-v4-pro")],
}
state = {"task_result": None, "task_fingerprint": None, "transformation_result": None, "transformation_fingerprint": None, "runs": []}

def _provider(label, allow_live):
    """Return the selected provider name after explicit live-call opt-in."""
    if not allow_live:
        raise RuntimeError("Live provider calls are disabled. Select the opt-in checkbox before running.")
    env_name = PROVIDER_ENV[label]
    if not os.getenv(env_name):
        raise RuntimeError(f"{env_name} is not set; configure credentials in the environment.")
    return label.lower()

def _fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

def _set_panel(panel, *items):
    """Replace one panel value; never publish callback output via IOPub capture."""
    panel.value = "".join(
        item.to_html(index=False, escape=True)
        if isinstance(item, pd.DataFrame) else str(item)
        for item in items
    )

def _error(panel, exc):
    _set_panel(panel, f"<p><strong>Run stopped:</strong> {escape(type(exc).__name__)} — {escape(str(exc))}</p>")


## 2. Example cases

Cases are illustrative assumptions, not validated estimates. Add a case by adding the same fields to `TASK_CASES`; use **Load case** to copy it into editable widgets.

In [2]:
TASK_CASES = {
    "invoice_review": {
        "label": "Supplier invoice review",
        "task_name": "Review and approve a supplier invoice",
        "task_description": "You work in accounts payable at a mid-sized manufacturing company. A supplier has sent an invoice for raw materials delivered to your factory. It lists the supplier's details, purchase order number, items, quantities, prices, taxes, and payment terms. Your task is to check that the invoice is genuine, the numbers are right, and everything matches what the company ordered and received. You might run into things like a wrong price, a missing purchase order, quantities that don't match, a duplicate invoice, or charges you don't recognize. If something doesn't add up, you'll need to check with purchasing, the warehouse, or the supplier. Once everything checks out and any issues are sorted out, approve the invoice for payment under company policy.",
        "domain": "accounts_payable",
    },
    "customer_support": {
        "label": "SaaS customer support",
        "task_name": "Triage a software customer support ticket",
        "task_description": "You work in customer support at a B2B software company. A customer sends in a ticket saying several employees can't open their reporting dashboard after a recent software update. They've included a short description, screenshots, and roughly when the problem started. You can look up their subscription, account history, earlier support tickets, product documentation, and known incidents. Your job is to figure out what's going on, how serious it is, and how it's affecting their business. Check whether there's already a known fix, then decide whether you can help them directly or need to pass the issue to technical support or engineering. Send the customer a response explaining what happens next.",
        "domain": "customer_support",
    },
    "production_scheduling": {
        "label": "Manufacturing production scheduling",
        "task_name": "Prepare a weekly production schedule",
        "task_description": "You're a production planner at a packaged food factory, and you need to put together next week's schedule for several packaging lines. You have customer orders, demand forecasts, inventory levels, equipment availability, staff schedules, and planned maintenance times to work with. Products take different amounts of time to run and need different packaging setups. Switching products can also mean stopping to clean and reset the equipment. Your task is to build a schedule the factory can actually deliver, taking into account deadlines, line capacity, available materials, staffing, and production priorities. If there isn't enough capacity or some materials won't be available, work through the options with production management before sending the schedule to the factory floor.",
        "domain": "operations",
    },
    "claims_triage": {
        "label": "Auto insurance claims triage",
        "task_name": "Triage a personal auto collision claim",
        "task_description": "You handle new auto insurance claims. A policyholder reports a two-car collision at an intersection and sends in their account of what happened, vehicle details, damage photos, the other driver's contact and insurance information, and a police report number. Some details might be missing or conflicting, and there may not be a repair estimate yet. Your task is to check the claim against the policy and available records, see whether there's enough information to move it forward, flag missing documents or possible coverage issues, and send it to the right claims team. Straightforward claims can go through the usual process. Claims involving injuries, conflicting accounts, signs of possible fraud, or complicated coverage questions need specialist attention. You're getting the claim ready for an adjuster, who will make the final coverage and liability decisions.",
        "domain": "insurance",
    },
}

assert TASK_CASES

## 3. Shared request-building and execution functions

These functions accept explicit values, do not read widgets, and do not display or mutate notebook state.

In [3]:
def build_task_request(inputs):
    return TaskDecompositionRequest(task=TaskReference(task_name=inputs["task_name"], task_description=inputs["task_description"]), task_context={"domain": inputs["domain"]})

def execute_task(inputs, provider, model=None, max_output_tokens=None):
    return decompose_task(build_task_request(inputs), provider, model=model, max_output_tokens=max_output_tokens)

def execute_transformation(task_result, inputs, provider, model=None, max_output_tokens=None, mode="normalized"):
    return estimate_transformation(task_result, provider, transformation_context={"domain": inputs["domain"]}, model=model, max_output_tokens=max_output_tokens)

def _allocation_percent(allocation):
    return f"{allocation.weight_ratio:.2%}" if allocation and allocation.weight_ratio is not None else None

def task_table(result):
    allocations = {a.subtask_id: a for a in result.baseline_effort_allocations}
    rows = []
    for subtask in sorted(result.operational_decomposition.operational_subtasks, key=lambda item: item.sequence_index):
        allocation = allocations.get(subtask.subtask_id)
        rows.append({
            "sequence": subtask.sequence_index,
            "subtask_id": subtask.subtask_id,
            "subtask": subtask.subtask_name,
            "description": subtask.description,
            "relative_effort_weight": subtask.relative_effort_weight,
            "normalized_effort": _allocation_percent(allocation),
            "effort_rationale": subtask.effort_rationale,
        })
    return pd.DataFrame(rows)

def _display_quantity(value):
    if value is None:
        return "Unavailable"
    parts = [str(value.value)]
    basis = getattr(value, "basis", None)
    if basis is not None:
        parts.append(f"basis={getattr(basis, 'value', basis)}")
    unit = getattr(value, "unit", None)
    time_basis = getattr(value, "time_basis", None)
    if unit is not None:
        parts.append(f"unit={getattr(unit, 'value', unit)}")
    if time_basis is not None:
        parts.append(f"time_basis={getattr(time_basis, 'value', time_basis)}")
    return "; ".join(parts)

def _display_minutes(value):
    if value is None:
        return "Unavailable"
    unit = getattr(value.unit, "value", value.unit)
    if unit == "hours":
        return f"{value.value * 60} minutes"
    if unit == "days":
        return f"{value.value * 1440} minutes"
    return f"{value.value} {unit} (not convertible to minutes)"

def _display_range(value):
    if value is None:
        return "Unavailable"
    return f"typical={value.typical} (low={value.low}, high={value.high})"

def _added_detail_table(result):
    details = getattr(result, "added_work_estimate_details", ())
    if not details:
        return pd.DataFrame([{"status": "Per-activity calculation details unavailable for this result"}])
    return pd.DataFrame([{
        "activity_id": detail.work_id,
        "activity": detail.workload_name,
        "description": detail.description or "Unavailable",
        "category": detail.category.value,
        "expected_occurrences_per_parent_output": _display_range(detail.occurrences_per_parent_execution),
        "relative_effort_per_occurrence": _display_range(detail.relative_effort_per_occurrence),
        "reference_source": detail.reference_source,
        "reference_subtask_id": detail.reference_subtask_id or "Unavailable",
        "external_reference_activity": detail.external_reference_activity or "Unavailable",
        "reference_effort": _display_quantity(detail.reference_effort),
        "reference_occurrences_per_parent_output": _display_range(detail.reference_occurrences_per_parent_execution),
        "reference_effort_per_occurrence": _display_quantity(detail.reference_effort_per_occurrence),
        "calculated_added_effort_per_parent_output": _display_quantity(detail.calculated_added_effort),
        "estimate_basis": detail.estimate_basis,
        "rationale": detail.rationale,
        "provenance": ", ".join(f"{ref.kind}:{ref.value}" for ref in detail.provenance_refs) or "Unavailable",
    } for detail in details])

def transformation_table(result, estimates):
    rows = [{"subtask_id": item.subtask_id, "subtask": item.subtask_name, "classification": result.retain_remove_classification.classified_subtasks[index].classification.value, "baseline_work_units": item.baseline_work_units, "retained_work_units": item.retained_work_units, "gross_removed_work_units": item.gross_removed_work_units, "estimate_source": item.estimate_source, "assumption": item.assumption, "rationale": item.rationale} for index, item in enumerate(estimates)]
    estimate_table = pd.DataFrame(rows)
    added_rows = [{"work_id": row.work_id, "category": row.category.value, "source_removed_subtask_ids": ", ".join(row.source_removed_subtask_ids), "estimate_basis": row.estimate.estimate_basis if row.estimate else None, "rationale": row.estimate.rationale if row.estimate else None} for row in result.added_work_classification.added_work_rows]
    added_table = pd.DataFrame(added_rows) if added_rows else pd.DataFrame([{"status": "No legacy added-work rows; activity-based estimates are authoritative"}])
    accounting = result.accounting
    retained = sum(item.retained_work_units for item in estimates)
    added = accounting.added_human_work_ratio * 100
    estimated_support = result.support_work_resolution.estimated_support_work if result.support_work_resolution else None
    interaction = estimated_support.ai_interaction.value * 100 if estimated_support else "Unavailable"
    verification = estimated_support.ai_verification.value * 100 if estimated_support else "Unavailable"
    totals = pd.DataFrame([{"normalized_baseline_W0": 100, "retained_work_units": retained, "gross_removed_work_units": sum(item.gross_removed_work_units for item in estimates), "added_work_units": added, "normalized_post_transform_W1": accounting.w1 * 100, "effect": accounting.effect.value, "assumption": "100 units are normalized relative work, not hours or measured productivity."}])
    reconciliation = pd.DataFrame([{"calculation": "Added AI interaction + added AI verification", "value": f"{interaction} + {verification} = {added} normalized baseline-relative units per parent output"}, {"calculation": "Retained human effort + added human effort", "value": f"{retained} + {added} = {accounting.w1 * 100} normalized baseline-relative units per parent output (canonical W1)"}])
    detail_table = _added_detail_table(result)
    synthesis = result.agentic_task_synthesis
    agentic_rows = [{"agentic_task_id": task.agentic_task_id, "name": task.agentic_task_name, "replaces": ", ".join(task.source_subtask_ids), "responsibility": task.description, "inputs": ", ".join(task.required_inputs), "outputs": ", ".join(task.expected_outputs)} for task in synthesis.agentic_tasks] if synthesis else []
    agentic_table = pd.DataFrame(agentic_rows) if agentic_rows else pd.DataFrame([{"status": "Synthesis unavailable or no agentic tasks returned"}])
    activity_model = result.human_agent_activity_modeling
    activity_rows = [{"activity_id": activity.activity_id, "name": activity.activity_name, "category": activity.category.value, "agentic_task_ids": ", ".join(activity.agentic_task_ids), "trigger": activity.trigger, "expected_action": activity.expected_action} for activity in activity_model.activities] if activity_model else []
    activity_table = pd.DataFrame(activity_rows) if activity_rows else pd.DataFrame([{"status": "Activity modeling unavailable or no activities returned"}])
    activity_effort = result.activity_effort_estimation
    activity_by_id = {activity.activity_id: activity for activity in activity_model.activities} if activity_model else {}
    effort_rows = [{"activity_id": item.activity_id, "activity": activity_by_id[item.activity_id].activity_name if item.activity_id in activity_by_id else item.activity_id, "category": activity_by_id[item.activity_id].category.value if item.activity_id in activity_by_id else "Unavailable", "expected_occurrences_per_parent_output": item.expected_occurrences_per_parent_output, "minutes_per_occurrence": _display_minutes(item.active_effort_per_occurrence), "expected_minutes_per_parent_output": _display_minutes(item.expected_effort_per_parent_output), "rationale": item.rationale, "assumptions": "; ".join(item.assumptions) or "None recorded"} for item in activity_effort.activity_effort_estimates] if activity_effort else []
    effort_table = pd.DataFrame(effort_rows) if effort_rows else pd.DataFrame([{"status": "Activity-based effort estimation unavailable or no activities returned"}])
    calibration_quantity = accounting.baseline_human_effort or (activity_effort.estimated_baseline_effort_per_parent_output if activity_effort else None)
    calibration = _display_minutes(calibration_quantity)
    calibration_source = "supplied" if accounting.baseline_human_effort else ("estimated" if activity_effort and activity_effort.estimated_baseline_effort_per_parent_output else "unavailable")
    calibration_rationale = activity_effort.baseline_rationale if activity_effort and activity_effort.baseline_rationale else "No estimated calibration rationale recorded"
    calibration_assumptions = "; ".join(activity_effort.baseline_assumptions) if activity_effort and activity_effort.baseline_assumptions else "None recorded"
    calibration_provenance = "Caller-supplied accounting input" if calibration_source == "supplied" else "; ".join(f"{item.provider_id}:{item.model_id}" for item in result.provider_provenance if getattr(item.stage, "value", item.stage) == "activity_effort_estimation") or "Provider-stage provenance unavailable"
    category_totals = {}
    if activity_effort:
        for item in activity_effort.activity_effort_estimates:
            category = activity_by_id[item.activity_id].category.value
            current = category_totals.get(category)
            category_totals[category] = item.expected_effort_per_parent_output.model_copy(update={"value": (current.value if current else Decimal(0)) + item.expected_effort_per_parent_output.value})
    category_table = pd.DataFrame([{"category": category, "expected_effort_per_parent_output": _display_minutes(value)} for category, value in category_totals.items()]) if category_totals else pd.DataFrame([{"status": "No activity-level category totals available"}])
    method = getattr(result, "estimation_method", "legacy_reference_based")
    if method == "activity_based":
        category_summary = "; ".join(f"{category}: {_display_minutes(value)}" for category, value in category_totals.items()) or "No modeled activities"
        reconciliation = pd.DataFrame([{"calculation": "Activity-level effort by category (reporting projection)", "value": category_summary}, {"calculation": "Canonical added human effort", "value": f"{accounting.added_human_work_ratio * 100} normalized units per parent output"}])
    denominator = "Parent output unit: one execution of the decomposed task. Occurrences are per parent output; normalized values are not minutes or hours."
    if method == "activity_based":
        legacy_sections = "<h4>Legacy added-work proposals</h4><p>Not run: activity-based estimates are authoritative for this transformation.</p><h4>Added-work calculation details (legacy reference-based)</h4><p>Not applicable: activity estimates are shown above and converted directly to canonical support work.</p>"
    else:
        legacy_sections = "<h4>Legacy added-work proposals</h4>" + added_table.to_html(index=False, escape=True) + "<h4>Added-work calculation details (legacy reference-based)</h4><p>" + denominator + "</p>" + detail_table.to_html(index=False, escape=True)
    accounting_summary = pd.DataFrame([{"normalized_W0": accounting.w0 * 100, "retained_baseline_effort": retained, "added_human_effort": accounting.added_human_work_ratio * 100, "normalized_W1": accounting.w1 * 100, "net_effort_reduction_ratio": accounting.net_substitution_ratio, "accounting_mode": accounting.accounting_mode.value}])
    calibration_html = "<h4>Baseline calibration</h4>" + pd.DataFrame([{"value": calibration, "source": calibration_source, "provenance": calibration_provenance, "rationale": calibration_rationale, "assumptions": calibration_assumptions}]).to_html(index=False, escape=True)
    return "<h4>Per-subtask normalized estimate</h4>" + estimate_table.to_html(index=False, escape=True) + "<h4>Agentic task synthesis (Phase 1)</h4>" + agentic_table.to_html(index=False, escape=True) + "<h4>Human-agent activities (Phase 2)</h4>" + activity_table.to_html(index=False, escape=True) + "<h4>Activity-based effort estimates (Phase 3)</h4><p>Estimation method: " + escape(method) + ".</p>" + effort_table.to_html(index=False, escape=True) + "<h4>Activity effort by category (reporting projection)</h4>" + category_table.to_html(index=False, escape=True) + calibration_html + legacy_sections + "<h4>Canonical accounting summary</h4>" + accounting_summary.to_html(index=False, escape=True) + "<h4>Added-work reconciliation</h4>" + reconciliation.to_html(index=False, escape=True) + "<h4>Accounting totals</h4>" + totals.to_html(index=False, escape=True)

def snapshot(stage, case_key, provider, inputs, result, estimates=()):
    stamp = datetime.now(timezone.utc).isoformat()
    return {"run_id": _fingerprint((stamp, stage, provider, inputs))[:12], "timestamp": stamp, "case_key": case_key or "edited", "provider": provider, "model": result.provider_provenance[-1].model_id if getattr(result, "provider_provenance", ()) else None, "stage": stage, "validation_status": "passed", "assumptions": list(getattr(result, "estimation_assumptions", ())), "inputs": deepcopy(inputs), "result": result.model_dump(mode="json") if hasattr(result, "model_dump") else result, "normalized_estimates": [item.model_dump(mode="json") for item in estimates]}

def persist_run(run):
    output_dir = Path("outputs")
    output_dir.mkdir(exist_ok=True)
    artifact = {key: value for key, value in run.items() if key != "inputs"}
    artifact["input_case_id"] = run["case_key"]
    path = output_dir / f"{run['run_id']}.json"
    path.write_text(json.dumps(artifact, indent=2, default=str), encoding="utf-8")
    return path


## 4. Interactive controls and stage-by-stage execution

Select one of the predefined cases and run the stages explicitly.

In [4]:
# Retire this UI before rebuilding it when this cell is rerun.
for old_button, old_callback in globals().get("_ui_click_bindings", []):
    old_button.on_click(old_callback, remove=True)
for old_widget, old_callback in globals().get("_ui_observer_bindings", []):
    old_widget.unobserve(old_callback, names="value")
for old_widget in reversed(globals().get("_ui_widgets", [])):
    old_widget.close()
clear_output(wait=True)
_ui_click_bindings = []
_ui_observer_bindings = []

provider_widget = widgets.Dropdown(options=list(PROVIDER_ENV), value="OpenAI", description="Provider:", style={"description_width": "180px"})
model_widget = widgets.Dropdown(options=MODEL_OPTIONS["OpenAI"], value="gpt-6-luna", description="Model:", style={"description_width": "180px"})
allow_live_widget = widgets.Checkbox(value=False, description="Allow paid/networked LLM calls")
case_widget = widgets.Dropdown(options=[(v["label"], k) for k, v in TASK_CASES.items()], description="Example case:", style={"description_width": "180px"})
task_name_widget = widgets.HTML()
run_status_widget = widgets.HTML(value="<b>Status:</b> Idle")
task_review_widget = widgets.Textarea(description="Human review notes:", layout=widgets.Layout(width="100%", height="60px"), style={"description_width": "180px"})
task_output, transform_output, end_output, history_output = (
    widgets.HTML(layout=widgets.Layout(width="100%", overflow="auto"))
    for _ in range(4)
)
run_task_button = widgets.Button(description="Run task decomposition", button_style="primary")
run_transform_button = widgets.Button(description="Run transformation decomposition", button_style="primary")
run_end_button = widgets.Button(description="Run estimated end-to-end decomposition", button_style="warning")
run_history_button = widgets.Button(description="Compare saved runs")

def current_task_inputs():
    case = TASK_CASES[case_widget.value]
    return {key: case[key] for key in ("task_name", "task_description", "domain")}
def current_inputs():
    return current_task_inputs()
def invalidate_task(*_):
    state.update(task_result=None, task_fingerprint=None, transformation_result=None, transformation_fingerprint=None)
    task_output.value = ""
    transform_output.value = ""
    end_output.value = ""
def invalidate_transformation(*_):
    state.update(transformation_result=None, transformation_fingerprint=None)
def update_model_options(change):
    provider = change["new"]
    model_widget.options = MODEL_OPTIONS[provider]
    model_widget.value = ""
    invalidate_task()
def set_run_status(message):
    run_status_widget.value = f"<b>Status:</b> {escape(message)}"
    run_status_widget.send_state("value")
def run_task(_=None):
    # This is an in-flight lock, not a cache: intentional repeated runs are allowed.
    if state.get("task_running"):
        return
    state["task_running"] = True
    run_task_button.disabled = True
    inputs = current_task_inputs()
    provider_label = provider_widget.value
    case_key = case_widget.value
    try:
        started = time.perf_counter()
        set_run_status(f"Running task decomposition — LLM call 1/2 in progress ({provider_label})...")
        result = execute_task(inputs, _provider(provider_label, allow_live_widget.value), model_widget.value.strip() or None, max_output_tokens=16000)
        set_run_status(f"Task decomposition complete — 2/2 LLM calls finished in {time.perf_counter() - started:.1f}s")
        state.update(task_result=result, task_fingerprint=_fingerprint(inputs), transformation_result=None, transformation_fingerprint=None)
        state["runs"].append(snapshot("task", case_key, provider_label, inputs, result))
        _set_panel(
            task_output,
            "<h4>Task decomposition</h4>",
            task_table(result),
        )
        transform_output.value = ""
        end_output.value = ""
    except Exception as exc:
        set_run_status(f"Task decomposition failed: {type(exc).__name__}")
        state.update(task_result=None, task_fingerprint=None, transformation_result=None, transformation_fingerprint=None)
        _error(task_output, exc)
    finally:
        state["task_running"] = False
        run_task_button.disabled = False
def run_transform(_=None):
    inputs = current_inputs()
    if state["task_result"] is None or state["task_fingerprint"] != _fingerprint(current_task_inputs()): return _error(transform_output, RuntimeError("Run task decomposition again: the task result is missing or stale."))
    try:
        started = time.perf_counter()
        set_run_status(f"Running transformation — LLM call 1/2 in progress ({provider_widget.value})...")
        result = execute_transformation(state["task_result"], inputs, _provider(provider_widget.value, allow_live_widget.value), model_widget.value.strip() or None, max_output_tokens=16000)
        estimates = result.normalized_work_estimates
        set_run_status(f"Transformation complete — 2/2 LLM calls finished in {time.perf_counter() - started:.1f}s")
        state.update(transformation_result=result, transformation_fingerprint=_fingerprint(inputs))
        run = snapshot("transformation", case_widget.value, provider_widget.value, inputs, result, estimates)
        state["runs"].append(run)
        persist_run(run)
        _set_panel(transform_output, transformation_table(result, estimates))
    except Exception as exc:
        set_run_status(f"Transformation failed: {type(exc).__name__}")
        _error(transform_output, exc)
def run_end_to_end(_=None):
    inputs = current_inputs()
    try:
        started = time.perf_counter()
        set_run_status(f"Running end-to-end decomposition — LLM call 1/4 in progress ({provider_widget.value})...")
        provider = _provider(provider_widget.value, allow_live_widget.value)
        task_result = execute_task(inputs, provider, model_widget.value.strip() or None, max_output_tokens=16000)
        result = execute_transformation(task_result, inputs, provider, model_widget.value.strip() or None, max_output_tokens=16000)
        estimates = result.normalized_work_estimates
        set_run_status(f"End-to-end decomposition complete — 4/4 LLM calls finished in {time.perf_counter() - started:.1f}s")
        run = snapshot("end_to_end", case_widget.value, provider_widget.value, inputs, result, estimates)
        state["runs"].append(run)
        persist_run(run)
        _set_panel(end_output, transformation_table(result, estimates))
    except Exception as exc:
        set_run_status(f"End-to-end decomposition failed: {type(exc).__name__}")
        _error(end_output, exc)
def show_history(_=None):
    rows = []
    for run in state["runs"]:
        result = run["result"]
        task = result.get("task_decomposition", result) if isinstance(result, dict) else {}
        subtasks = task.get("operational_decomposition", {}).get("operational_subtasks")
        rows.append({"run_id": run["run_id"], "case": run["case_key"],
                     "provider": run["provider"], "stage": run["stage"],
                     "subtasks": len(subtasks) if subtasks is not None else None})
    _set_panel(history_output, pd.DataFrame(rows))

def update_task_name(_=None):
    case = TASK_CASES[case_widget.value]
    task_name_widget.value = (
        f"<h3>{escape(case['task_name'])}</h3>"
        f"<p>{escape(case['task_description'])}</p>"
    )

_ui_observer_bindings = [
    (case_widget, invalidate_task),
    (provider_widget, update_model_options),
    (model_widget, invalidate_task),
    (case_widget, update_task_name),
]
for widget, callback in _ui_observer_bindings:
    widget.observe(callback, names="value")
_ui_click_bindings = [
    (run_task_button, run_task),
    (run_transform_button, run_transform),
    (run_end_button, run_end_to_end),
    (run_history_button, show_history),
]
for button, callback in _ui_click_bindings:
    button.on_click(callback)
update_task_name()
# Review notes are mounted once, outside any result panel.
ui = widgets.VBox([
    provider_widget, model_widget, allow_live_widget, case_widget, task_name_widget, run_status_widget,
    run_task_button, task_output, widgets.HTML("<h3>Human review</h3>"), task_review_widget,
    run_transform_button, transform_output, run_end_button, end_output,
    run_history_button, history_output,
])
_ui_widgets = [ui, *ui.children]
invalidate_task()
display(ui)


## 5. Added-work estimation and support-work resolution

The transformation result contains RETAIN/REMOVE classifications, Phase 1 agentic task synthesis, Phase 2 human-agent activities, Phase 3 activity-based effort estimates when supported, and the canonical support-work resolution. This notebook displays those results; it does not reimplement support-work formulas.

## 6. Deterministic accounting

The notebook derives a 100-unit normalized baseline from task relative-effort weights, then applies canonical deterministic accounting. Activity estimates are per parent output and are not interpreted as minutes or hours unless the result includes an explicit calibration.

In [ ]:
display(Markdown("Transformation estimates use a normalized 100-unit baseline. These units are comparative, not measured hours or productivity."))


## 7. End-to-end normalized estimation execution

This explicit button composes task decomposition with provider-assisted normalized transformation estimation. It uses the current editable inputs and does not reuse a stale transformation result. The output shows which estimation method was used and keeps legacy reference-based results distinguishable from activity-based results.

## 8. Provider comparison and repeatability

Run one provider at a time with the explicit task button, then inspect the preserved `state['runs']` history. Counts are operational summaries only and are not semantic quality scores.